In [11]:
import pandas as pd

pd.set_option('display.max_columns', 10)

In [12]:
import duckdb

con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,32
1,222,2x2x2 Cube,30
2,clock,Clock,28
3,pyram,Pyraminx,26
4,minx,Megaminx,26
5,444,4x4x4 Cube,25
6,333oh,3x3x3 One-Handed,22
7,skewb,Skewb,21
8,333bf,3x3x3 Blindfolded,21
9,555,5x5x5 Cube,19


In [13]:
con = duckdb.connect()

df_camp_by_year = con.execute("""
SELECT
    year,
    COUNT(*) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t')
WHERE
    country_id = 'Brazil'
    AND cancelled = 0
GROUP BY year
ORDER BY year
""").df()


In [14]:
import duckdb

con = duckdb.connect()

df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [15]:
df_final.head()

,event_id,event_name,num_competitions,year,total_competitions
0,444,4x4x4 Cube,1,2007,1
1,555,5x5x5 Cube,1,2007,1
2,333,3x3x3 Cube,1,2007,1
3,222,2x2x2 Cube,1,2007,1
4,minx,Megaminx,1,2007,1


In [16]:
import plotly.graph_objects as go

def plot_events_over_time(df):

    fig = go.Figure()

    # linhas por modalidade
    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["num_competitions"],
                mode="lines+markers",
                name=event_name,
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Competições: %{y}<extra></extra>"
                )
            )
        )

    # linha de total de competições
    total_df = (
        df[["year", "total_competitions"]]
        .drop_duplicates()
        .sort_values("year")
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["year"],
            y=total_df["total_competitions"],
            mode="lines",
            name="Total de Competições",
            line=dict(width=5),
            hovertemplate=(
                "<b>Total de Competições</b><br>"
                "Ano: %{x}<br>"
                "Total: %{y}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title="Quantidade de Competições por Modalidade no Brasil",
        xaxis_title="Ano",
        yaxis_title="Número de Competições",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Modalidade",
        height=700
    )

    fig.write_html(
    "./index.html",
    include_plotlyjs="cdn")

    fig.show()

In [17]:
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]

In [18]:
df_final.sort_values(by='event_name', inplace=True)

In [19]:
plot_events_over_time(df_final)